# 06 - RAG Chunking en Vector Storage Deep Dive (Nederlandse Editie)

Deze notebook demonstreert de **volledige RAG (Retrieval-Augmented Generation) pipeline** van document chunking tot vector storage en semantisch zoeken, met voorbeelden specifiek voor Nederland.

## 🇳🇱 Nederlandse Context
We bouwen een RAG systeem met:
- Nederlandse bedrijfsdocumentatie
- Azure-resources in West Europe (Amsterdam)
- AVG/GDPR compliance overwegingen
- Nederlandse taalondersteuning

## 🧠 Wat Je Zult Leren
- **Document Chunking**: Grote teksten opdelen in doorzoekbare segmenten
- **Embedding Creatie**: Tekstchunks omzetten naar vectorrepresentaties
- **Vector Opslag**: Embeddings met metadata opslaan voor retrieval
- **Semantisch Zoeken**: Relevante chunks vinden met cosine similarity
- **RAG Integratie**: Retrieval combineren met generatie

## 📊 RAG Pipeline Overzicht

```
📄 Document → 🔪 Chunking → 🧠 Embedding → 🗃️ Vector Store → 🔍 Zoeken → 🤖 Generatie
```

In [ ]:
import os
import json
import numpy as np
from openai import AzureOpenAI
from dotenv import load_dotenv
import re
from typing import List, Dict, Tuple
from sklearn.metrics.pairwise import cosine_similarity

load_dotenv()

# Azure OpenAI Configuratie
client = AzureOpenAI(
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    api_version="2024-05-01-preview"
)

gpt_deployment = os.getenv("AZURE_OPENAI_GPT4O_DEPLOYMENT_NAME")
embeddings_deployment = os.getenv("AZURE_OPENAI_EMBEDDINGS_DEPLOYMENT_NAME")

print("✅ Azure OpenAI client geïnitialiseerd voor RAG pipeline")
print(f"🤖 GPT Model: {gpt_deployment}")
print(f"🧠 Embeddings Model: {embeddings_deployment}")
print(f"📐 Verwachte embedding dimensies: 3072 (text-embedding-3-large)")

## 📄 Stap 1: Maak Nederlandse Voorbeelddocumenten

Laten we realistische Nederlandse bedrijfsdocumenten maken die we zullen chunken en opslaan.

In [ ]:
def create_dutch_documents() -> Dict[str, str]:
    """Maak Nederlandse voorbeelddocumenten voor RAG demonstratie"""
    
    documents = {
        "azure_prestaties": """
Azure OpenAI Prestatie Best Practices voor Nederlandse Organisaties: Gebruik connection pooling om latentie te verminderen.
Implementeer exponentiële backoff voor rate limiting. Cache responses waar mogelijk.
Monitor tokengebruik om kosten te optimaliseren. Gebruik streaming voor real-time applicaties.
Overweeg modelselectie op basis van uw use case - GPT-4o voor complexe redenering,
GPT-3.5 voor snellere responses. De West Europe regio (Amsterdam) biedt de laagste latentie voor Nederlandse gebruikers.
        """.strip(),
        
        "azure_beveiliging": """
Azure OpenAI Beveiligingsrichtlijnen voor Nederland: Gebruik Azure AD-authenticatie en beheerde identiteiten.
Implementeer passende RBAC-controles. Schakel contentfiltering in om schadelijke content te voorkomen.
Gebruik private endpoints voor netwerkisolatie. Roteer regelmatig API-sleutels.
Monitor toegangslogs voor verdachte activiteit. Implementeer invoervalidatie en sanitization.
Gebruik Azure Key Vault voor geheimenbeheer. Schakel diagnostische logging in voor audit trails.
Stel waarschuwingsregels in voor ongebruikelijke gebruikspatronen. Voldoe aan AVG/GDPR vereisten.
        """.strip(),
        
        "azure_kosten": """
Azure OpenAI Kostenbeheer voor Nederlandse Bedrijven: Monitor tokengebruik met Azure Monitor.
Stel facturatiewaarschuwingen en quota's in. Gebruik prompt engineering om tokenconsumptie te verminderen.
Kies geschikte modellen voor uw use case. Implementeer caching-strategieën.
Gebruik batchverwerking waar mogelijk. Overweeg kleinere modellen voor eenvoudige taken.
Optimaliseer prompts om invoer- en uitvoertokens te verminderen. Gebruik streaming om waargenomen prestaties te verbeteren.
Implementeer rate limiting om kosten te beheersen. Monitor kosten over verschillende deployments.
Budgetten in euro's instellen voor Nederlandse boekhouding.
        """.strip(),
        
        "ns_reizen": """
NS Treinreizen Informatie voor Zakelijke Reizigers: De intercity tussen Amsterdam en Rotterdam duurt 40 minuten.
Boek via NS Zakelijk voor gecentraliseerde facturatie. Gebruik NS Flex voor flexibele prijzen.
Spitsuren zijn van 07:00-09:00 en 16:30-18:30. Reis buiten spits voor 40% korting met Dal Voordeel.
De OV-chipkaart is verplicht voor alle treinreizen. Eerste klasse biedt meer werkruimte en rust.
WiFi is beschikbaar op de meeste intercity treinen. Reserveer stiltecoupe voor geconcentreerd werken.
Schiphol Airport is direct verbonden met Amsterdam Centraal (15 minuten).
        """.strip()
    }
    
    print("📄 Nederlandse documenten gemaakt:")
    for doc_id, content in documents.items():
        print(f"   📋 {doc_id}: {len(content)} karakters")
    
    return documents

# Maak onze kennisbank
documents = create_dutch_documents()

## 🔪 Stap 2: Document Chunking Strategieën

Laten we verschillende chunking strategieën implementeren en zien hoe ze retrieval kwaliteit beïnvloeden.

In [ ]:
class DocumentChunker:
    """Implementeert verschillende document chunking strategieën"""
    
    def __init__(self, chunk_size: int = 500, overlap: int = 50):
        self.chunk_size = chunk_size
        self.overlap = overlap
    
    def sentence_based_chunking(self, text: str) -> List[str]:
        """Splits tekst op zinnen met configureerbare groepering"""
        # Split op zinnen (basis aanpak)
        sentences = re.split(r'[.!?]+', text)
        sentences = [s.strip() for s in sentences if s.strip()]
        
        chunks = []
        current_chunk = []
        current_length = 0
        
        for sentence in sentences:
            sentence_length = len(sentence)
            
            if current_length + sentence_length > self.chunk_size and current_chunk:
                chunks.append('. '.join(current_chunk) + '.')
                overlap_sentences = current_chunk[-1:] if self.overlap > 0 else []
                current_chunk = overlap_sentences + [sentence]
                current_length = sum(len(s) for s in current_chunk)
            else:
                current_chunk.append(sentence)
                current_length += sentence_length
        
        if current_chunk:
            chunks.append('. '.join(current_chunk) + '.')
        
        return chunks

# Demonstreer chunking
chunker = DocumentChunker(chunk_size=300, overlap=50)

print("🔪 CHUNKING STRATEGIE DEMONSTRATIE")
print("=" * 60)

sample_text = documents["azure_prestaties"]
chunks = chunker.sentence_based_chunking(sample_text)

print(f"📄 Originele tekst lengte: {len(sample_text)} karakters")
print(f"📋 Aantal chunks: {len(chunks)}")

for i, chunk in enumerate(chunks, 1):
    print(f"\n   Chunk {i}: {chunk[:80]}... ({len(chunk)} karakters)")

## 🧠 Stap 3: Genereer Embeddings voor Chunks

Nu converteren we onze tekstchunks naar vector embeddings met Azure OpenAI.

In [ ]:
class EmbeddingGenerator:
    """Handelt embedding generatie af met batching en error handling"""
    
    def __init__(self, client: AzureOpenAI, deployment_name: str):
        self.client = client
        self.deployment_name = deployment_name
        self.embedding_cache = {}
    
    def generate_embedding(self, text: str) -> List[float]:
        """Genereer embedding voor een enkele tekst"""
        
        if text in self.embedding_cache:
            return self.embedding_cache[text]
        
        try:
            response = self.client.embeddings.create(
                model=self.deployment_name,
                input=text.replace('\n', ' ').strip()
            )
            
            embedding = response.data[0].embedding
            self.embedding_cache[text] = embedding
            
            return embedding
            
        except Exception as e:
            print(f"❌ Fout bij embedding generatie: {e}")
            return None

def process_documents_to_embeddings():
    """Volledige pipeline: documenten → chunks → embeddings"""
    
    print("🧠 EMBEDDING GENERATIE PIPELINE")
    print("=" * 60)
    
    chunker = DocumentChunker(chunk_size=300, overlap=50)
    embedder = EmbeddingGenerator(client, embeddings_deployment)
    
    all_chunks = []
    
    for doc_id, content in documents.items():
        print(f"\n📄 Verwerken document: {doc_id}")
        
        chunks = chunker.sentence_based_chunking(content)
        print(f"   🔪 {len(chunks)} chunks gegenereerd")
        
        for i, chunk in enumerate(chunks):
            chunk_data = {
                "text": chunk,
                "document_id": doc_id,
                "chunk_index": i,
                "char_count": len(chunk),
                "source": "nederlandse_kennisbank"
            }
            all_chunks.append(chunk_data)
    
    print(f"\n📊 Totaal chunks te embedden: {len(all_chunks)}")
    
    print(f"\n🧠 Genereren embeddings...")
    embedded_chunks = []
    
    for chunk_data in all_chunks:
        embedding = embedder.generate_embedding(chunk_data["text"])
        if embedding:
            chunk_data["embedding"] = embedding
            chunk_data["embedding_dims"] = len(embedding)
            embedded_chunks.append(chunk_data)
            print(f"   ✅ Embedding gegenereerd: {len(embedding)} dimensies")
    
    print(f"\n✅ Embedding generatie voltooid!")
    print(f"   📊 Embedded chunks: {len(embedded_chunks)}")
    
    return embedded_chunks

embedded_chunks = process_documents_to_embeddings()

## 🗃️ Stap 4: Maak Vector Store en Opslag

Laten we een eenvoudige in-memory vector store implementeren om te begrijpen hoe vector databases werken.

In [ ]:
class SimpleVectorStore:
    """Eenvoudige in-memory vector store voor demonstratie"""
    
    def __init__(self):
        self.vectors = []
        self.metadata = []
        self.index_map = {}
    
    def add_vectors(self, embedded_chunks: List[Dict]):
        """Voeg embedded chunks toe aan de vector store"""
        
        print("🗃️ Vectors toevoegen aan store...")
        
        for chunk_data in embedded_chunks:
            if chunk_data.get("embedding"):
                chunk_id = f"{chunk_data['document_id']}_chunk_{chunk_data['chunk_index']}"
                
                vector_idx = len(self.vectors)
                self.vectors.append(np.array(chunk_data["embedding"]))
                
                metadata = {
                    "chunk_id": chunk_id,
                    "text": chunk_data["text"],
                    "document_id": chunk_data["document_id"],
                    "chunk_index": chunk_data["chunk_index"],
                    "char_count": chunk_data["char_count"],
                    "source": chunk_data.get("source", "onbekend")
                }
                self.metadata.append(metadata)
                self.index_map[chunk_id] = vector_idx
                
                print(f"   ✅ Toegevoegd: {chunk_id}")
        
        print(f"\n📊 Vector store status:")
        print(f"   📋 Totaal vectors: {len(self.vectors)}")
        print(f"   📐 Vector dimensies: {len(self.vectors[0]) if self.vectors else 0}")
    
    def similarity_search(self, query_embedding: List[float], top_k: int = 3) -> List[Dict]:
        """Vind meest vergelijkbare vectors met cosine similarity"""
        
        if not self.vectors:
            return []
        
        query_vector = np.array(query_embedding).reshape(1, -1)
        vector_matrix = np.vstack(self.vectors)
        
        similarities = cosine_similarity(query_vector, vector_matrix)[0]
        top_indices = np.argsort(similarities)[::-1][:top_k]
        
        results = []
        for idx in top_indices:
            result = {
                "similarity_score": float(similarities[idx]),
                "metadata": self.metadata[idx],
                "text": self.metadata[idx]["text"]
            }
            results.append(result)
        
        return results

# Maak en vul vector store
vector_store = SimpleVectorStore()
vector_store.add_vectors(embedded_chunks)

## 🔍 Stap 5: Implementeer Semantisch Zoeken

Laten we onze vector store testen met semantische zoekopdrachten in het Nederlands.

In [ ]:
class SemanticSearcher:
    """Handelt semantisch zoeken af met embedding generatie"""
    
    def __init__(self, vector_store: SimpleVectorStore, embedder: 'EmbeddingGenerator'):
        self.vector_store = vector_store
        self.embedder = embedder
    
    def search(self, query: str, top_k: int = 3) -> List[Dict]:
        """Zoek naar relevante documenten"""
        
        print(f"🔍 Zoeken naar: '{query}'")
        print("-" * 50)
        
        query_embedding = self.embedder.generate_embedding(query)
        
        if not query_embedding:
            print("❌ Kon query embedding niet genereren")
            return []
        
        results = self.vector_store.similarity_search(query_embedding, top_k)
        
        print(f"📊 {len(results)} resultaten gevonden:")
        for i, result in enumerate(results, 1):
            print(f"\n   🏆 Resultaat {i} (similarity: {result['similarity_score']:.4f})")
            print(f"   📄 Document: {result['metadata']['document_id']}")
            print(f"   📝 Tekst: {result['text'][:100]}...")
        
        return results

# Maak searcher
embedder = EmbeddingGenerator(client, embeddings_deployment)
searcher = SemanticSearcher(vector_store, embedder)

# Test met Nederlandse queries
print("\n" + "="*80 + "\n")
searcher.search("Hoe kan ik de kosten van Azure OpenAI verlagen?")

In [ ]:
print("\n" + "="*80 + "\n")
searcher.search("Wat zijn de beveiligingsrichtlijnen voor Azure?")

In [ ]:
print("\n" + "="*80 + "\n")
searcher.search("Hoe reis ik van Amsterdam naar Rotterdam met de trein?")

## 🤖 Stap 6: Complete RAG Systeem

Laten we alles samenbrengen tot een volledig RAG systeem.

In [ ]:
class DutchRAGSystem:
    """Volledig RAG systeem voor Nederlandse vragen"""
    
    def __init__(self, searcher: SemanticSearcher, client: AzureOpenAI, gpt_deployment: str):
        self.searcher = searcher
        self.client = client
        self.gpt_deployment = gpt_deployment
    
    def generate_answer(self, query: str, top_k: int = 3) -> Dict:
        """Genereer antwoord met RAG"""
        
        print(f"🤖 RAG: Beantwoorden van '{query}'")
        print("=" * 60)
        
        # Stap 1: Zoek relevante documenten
        search_results = self.searcher.search(query, top_k)
        
        if not search_results:
            return {"answer": "Ik kon geen relevante informatie vinden.", "sources": []}
        
        # Stap 2: Bouw context
        context_parts = []
        sources = []
        
        for i, result in enumerate(search_results, 1):
            context_parts.append(f"[Document {i}] {result['text']}")
            sources.append({
                "document_id": result["metadata"]["document_id"],
                "similarity_score": result["similarity_score"]
            })
        
        context = "\n\n".join(context_parts)
        
        # Stap 3: Genereer antwoord
        system_prompt = """Je bent een behulpzame assistent die vragen beantwoordt over Azure en Nederlandse bedrijfszaken.

Instructies:
- Gebruik ALLEEN de informatie uit de gegeven context
- Als de context niet genoeg informatie bevat, zeg dat duidelijk
- Geef specifiek, praktisch advies
- Verwijs naar de documenten met [Document 1], [Document 2], etc.
- Antwoord altijd in het Nederlands"""

        user_prompt = f"""Context uit kennisbank:
{context}

Vraag: {query}

Geef een uitgebreid antwoord gebaseerd op de context."""

        try:
            response = self.client.chat.completions.create(
                model=self.gpt_deployment,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt}
                ],
                max_tokens=1000,
                temperature=0.7
            )
            
            answer = response.choices[0].message.content
            
            print("\n📋 ANTWOORD:")
            print("-" * 40)
            print(answer)
            
            return {
                "query": query,
                "answer": answer,
                "sources": sources
            }
            
        except Exception as e:
            print(f"❌ Fout bij genereren antwoord: {e}")
            return {"answer": f"Fout: {e}", "sources": sources}

# Maak RAG systeem
rag_system = DutchRAGSystem(searcher, client, gpt_deployment)

# Test met Nederlandse vragen
print("\n" + "="*80 + "\n")
rag_system.generate_answer("Hoe kan ik de prestaties van mijn Azure OpenAI applicatie verbeteren?")

In [ ]:
print("\n" + "="*80 + "\n")
rag_system.generate_answer("Wat zijn de beveiligingsmaatregelen voor Azure OpenAI in Nederland?")

In [ ]:
print("\n" + "="*80 + "\n")
rag_system.generate_answer("Hoe reis ik het beste van Amsterdam naar Rotterdam voor een zakenreis?")

## 🎓 Wat Je Hebt Bereikt

✅ **Volledige RAG Pipeline**: Een volledig retrieval-augmented generation systeem van scratch gebouwd  
✅ **Document Chunking**: Verschillende chunking strategieën geïmplementeerd  
✅ **Vector Embeddings**: Semantische embeddings gegenereerd met Azure OpenAI  
✅ **Vector Opslag**: Een doorzoekbare vector database met metadata gemaakt  
✅ **Semantisch Zoeken**: Cosine similarity zoeken voor relevante documenten geïmplementeerd  
✅ **Antwoord Generatie**: Retrieval gecombineerd met LLM generatie voor uitgebreide antwoorden  
✅ **Nederlandse Context**: Alles toegepast met Nederlandse voorbeelden en taal  

## 🚀 Volgende Stappen

Nu je deze workshop hebt voltooid, ben je klaar om het **Model Context Protocol (MCP)** te verkennen!

👉 **[Microsoft MCP voor Beginners](https://github.com/microsoft/mcp-for-beginners)**

Daar leer je hoe MCP alle uitdagingen oplost die je hier hebt ervaren met:
- Dynamische tool-ontdekking
- Gestandaardiseerde protocollen
- Cross-applicatie context delen
- Naadloze integratie